# 08 | Alda-1: integrar a leitura humana e os perfis

**Papel nesta entrega:** Principal: caderno documentado do codigo de integracao utilizado

**Origem:** Entrega de 4 out. 2026; codigo analisar_integracao.py em Integracao_final_Alda1_e_texto_TCC.zip

Copia de publicacao organizada em 7 out. 2026. Saidas, contadores e metadados de sessao foram removidos; isso nao significa que a campanha original nao tenha sido executada. Nenhum resultado novo foi produzido na curadoria. O registro de proveniencia relaciona codigo, fonte e alteracoes de configuracao.

O codigo analitico da integracao foi preservado. Esta forma em caderno documentado e uma apresentacao posterior do script, nao uma nova campanha. Requer os 12 CSVs B1001 R1-R3 e os quatro ZIPs de leitura originais; veja dados_necessarios.md. A ficha corrigida acompanha os recursos.

# 04 | Integrar a avaliação humana aos perfis de Alda-1

**Pergunta:** a aparência do sinal, a intensidade agregada e A_Mito descrevem a mesma resposta?

Este caderno refaz a integração das 12 combinações local×réplica. Confere a chave, recupera os 28 DMSO de cada combinação e conserva a resposta humana. A dimensão principal de intensidade é `Cells_Intensity_MeanIntensity_Mito`; as 12 combinações de objeto (Cells/Cyto/Nuc) e canal são consultadas por significado, não escolhidas por concordarem com a ficha.

Não há ajuste de modelos, novos hits, exclusão de controles ou leitura de R4. A ficha corrigida é uma fonte, não uma classificação prevista automaticamente. As intensidades são medianas por poço de médias por objeto dos autores, e não brilho dos campos examinados.

**Necessário:** pacote final extraído no Drive e CSVs originais/resultados locais preservados em suas pastas. A célula de entrada falha com mensagem explícita se uma fonte estiver ausente.


## 1. Localizar fontes e abrir uma execução separada

Os arquivos são ligados a uma pasta temporária por nome, sem alterar seus bytes. A integridade é conferida antes da interpretação. Não há busca pelo composto para escolher arquivos favoráveis.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive
import os, json, shutil
from IPython.display import display
drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/mba_tcc')
ENTREGA = Path('/content/drive/MyDrive/mba_tcc/TCC_CellPainting_cadernos_repositorio/04_fontes_e_referencia/recursos')
FONTES = ENTREGA / 'fontes/integracao'
ROOT = Path('/content/integracao_Alda1_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
ROOT.mkdir(exist_ok=False)
OUT = ENTREGA / 'resultados_integracao' / ROOT.name
for pasta in ['tabelas','fontes','auditoria']:
    (OUT / pasta).mkdir(parents=True,exist_ok=False)
RUNS = {'IMTM':'IMTM_HepG2_20261003T185421_449156Z', 'FMP':'FMP_HepG2_20261003T181504_724095Z', 'MEDINA':'MEDINA_HepG2_20260930T235739_244794Z', 'USC':'USC_HepG2_20261001T105954_572551Z'}
for nome in ['Ficha_visual_simplificada_Alda1_corrigida.xlsx','Sintese_avaliacao_visual_Alda1_corrigida.md','chave_identidades_ABRIR_DEPOIS.csv','plano_pocos.csv','diagnosticos_pixels.csv']:
    p = FONTES / nome
    if not p.is_file(): raise FileNotFoundError(p)
    (ROOT / nome).symlink_to(p)
import pandas as pd
origens = pd.read_csv(ENTREGA / 'tabelas/alda1/fontes_perfis_B1001_R1_R3.csv')
for r in origens.itertuples():
    p = BASE / (r.site + '_HepG2') / r.arquivo
    if not p.is_file(): raise FileNotFoundError(p)
    (ROOT / r.arquivo).symlink_to(p)
for site, run in RUNS.items():
    tipo = 'didatico_padronizado' if site in ['IMTM','FMP'] else 'didatico_generalizavel'
    p = BASE / ('resultados_' + site + '_HepG2') / tipo / (run + '_leitura.zip')
    if not p.is_file(): raise FileNotFoundError(p)
    (ROOT / p.name).symlink_to(p)
print('Fontes localizadas. Saída:', OUT)

## 2. Definir medidas e utilitários

As medidas foram definidas por significado. Não trocamos o desfecho para concordar com a avaliação.

In [ ]:
"""Read-only descriptive integration. No fitted model, new hit rule, or R4 read."""
from pathlib import Path
import hashlib, io, json, zipfile, shutil
import numpy as np
import pandas as pd
from openpyxl import load_workbook
SITES = ['IMTM', 'FMP', 'MEDINA', 'USC']
REPS = ['R1', 'R2', 'R3']
FEATURES = [f'{ob}_Intensity_MeanIntensity_{ch}' for ob in ['Cells', 'Cyto',
    'Nuc'] for ch in ['Mito', 'AGP', 'DNA', 'ER']]
PRIMARY = 'Cells_Intensity_MeanIntensity_Mito'
RUNS = {'IMTM': 'IMTM_HepG2_20261003T185421_449156Z', 'FMP': 'FMP_HepG2_20261003T181504_724095Z',
    'MEDINA': 'MEDINA_HepG2_20260930T235739_244794Z', 'USC': 'USC_HepG2_20261001T105954_572551Z'}
SCORES = ['A_AGP', 'A_DNA', 'A_ER', 'A_Mito', 'A_Morfologia_Cells', 'A_Morfologia_Cyto', 'A_Morfologia_Nuc']

def digest(p):
    with Path(p).open('rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()

def readzip(z, name, **kw):
    return pd.read_csv(io.BytesIO(z.read(name)), **kw)

def save(name, df):
    df.to_csv(OUT / 'tabelas' / f'{name}.csv', index=False, encoding='utf-8-sig')

def require(c, msg):
    if not c:
        raise ValueError(msg)


## 3. Conferir as 36 identidades

A ficha humana, os códigos mascarados e o plano de poços precisam coincidir.

In [ ]:
tracked = {}
tests = []

def passed(name):
    tests.append({'check': name, 'status': 'passed'})
for name in ['Ficha_visual_simplificada_Alda1_corrigida.xlsx', 'Sintese_avaliacao_visual_Alda1_corrigida.md',
    'chave_identidades_ABRIR_DEPOIS.csv', 'plano_pocos.csv', 'diagnosticos_pixels.csv']:
    p = ROOT / name
    tracked[name] = digest(p)
    shutil.copy2(p, OUT / 'fontes' / name)
wb = load_workbook(ROOT / 'Ficha_visual_simplificada_Alda1_corrigida.xlsx', data_only=True)
visual = pd.DataFrame(list(wb['Preencher'].values)[7:19], columns=['site',
    'replica', 'avaliavel', 'Mito_visual', 'outros_visual', 'observacao_original']).fillna('')
require(len(visual) == 12 and visual[['site', 'replica']].drop_duplicates().shape[0] == 12, 'Unexpected visual rows')
require(visual.avaliavel.eq('Sim').all(), 'Corrected evaluability not reproduced')
key_x = pd.DataFrame(list(wb['Chave dos paineis'].values)[4:40], columns=['site',
    'replica', 'placa', 'papel_ficha', 'poco', 'sample_id'])
key = pd.read_csv(ROOT / 'chave_identidades_ABRIR_DEPOIS.csv', dtype=str)
require(len(key_x) == 36 and len(key) == 36, 'Expected 36 identifiers')
keys = ['site', 'replica', 'placa', 'poco', 'sample_id']
a = key_x[keys].sort_values(keys).reset_index(drop=True)
b = key[keys].sort_values(keys).reset_index(drop=True)
pd.testing.assert_frame_equal(a, b)
passed('36 workbook identities equal executed key')
key = key.merge(key_x, on=keys, validate='one_to_one')
require(key[key.papel_ficha.eq('Amostra')].composto.eq('EOS100315').all(), 'Case identity mismatch')
require(key[key.papel_ficha.str.startswith('DMSO')].composto.eq('DMSO').all(), 'Control identity mismatch')
passed('12 Alda-1 and 24 DMSO correctly labelled in key')
plan = pd.read_csv(ROOT / 'plano_pocos.csv')
plan.replica = plan.replica.astype(str)
require(set(plan.sample_id) == set(key.sample_id), 'Plan and key mismatch')
for col in ['site', 'placa', 'replica', 'poco', 'composto', 'tipo']:
    require(plan.set_index('sample_id')[col].sort_index().equals(key.set_index('sample_id')[col].sort_index()),
        f'Plan key mismatch {col}')
passed('Executed plan, workbook and mask key agree')
save('avaliacao_humana_preservada', visual)
save('identidades_conferidas', key)


## 4. Recuperar todos os DMSO e as intensidades

Cada grupo tem 28 DMSO e um poço de Alda-1. A referência usa todos os controles, não apenas os dois exibidos.

In [ ]:
wells = []
values = []
sources = []
schema = []
for site in SITES:
    pack = ROOT / (RUNS[site] + '_leitura.zip')
    tracked[pack.name] = digest(pack)
    with zipfile.ZipFile(pack) as z:
        sources_hash = json.loads(z.read('entradas_sha256.json'))
        scores = readzip(z, 'tabelas/05_escores_por_poco.csv')
        pars = readzip(z, 'tabelas/03_parametros_normalizacao.csv')
        scores = scores[scores.placa.eq('B1001') & scores.replica.isin(REPS)].copy()
        pars = pars[pars.placa.eq('B1001') & pars.replica.isin(REPS) & pars.medida.isin(FEATURES)].copy()
        pars.to_csv(OUT / 'fontes' / f'{site}_parametros_intensidade_congelados.csv', index=False)
        scores[scores.composto.isin(['DMSO', 'EOS100315'])].to_csv(OUT / 'fontes' / f'{site}_escores_fonte_B1001_R1_R3.csv',
            index=False)
        for rep in REPS:
            group = scores[scores.replica.eq(rep)].copy()
            names = group.arquivo.unique()
            require(len(names) == 1, 'Ambiguous input file')
            raw_path = ROOT / names[0]
            sha = digest(raw_path)
            tracked[raw_path.name] = sha
            hashes = [v for p, v in sources_hash.items() if Path(p).name == raw_path.name]
            require(len(hashes) == 1 and hashes[0] == sha, 'Raw input hash mismatch ' + raw_path.name)
            passed(f'{site}/{rep} raw CSV matches recorded SHA256')
            header = pd.read_csv(raw_path, nrows=0).columns.tolist()
            require(set(FEATURES) <= set(header), 'Missing semantic intensity feature')
            meta = [c for c in header if c.startswith('Metadata_')]
            raw = pd.read_csv(raw_path, usecols=meta + FEATURES)
            raw['poco'] = raw.Metadata_Well.astype(str)
            raw['linha_csv_reconstruida'] = np.arange(2, len(raw) + 2)
            require(raw.poco.is_unique, 'Duplicate raw wells')
            g = group.merge(raw, on='poco', validate='one_to_one')
            require(len(g) == len(group), 'Raw-score join lost wells')
            require((g.linha_csv == g.linha_csv_reconstruida).all(), 'CSV line disagreement')
            require((g.n_objetos == g.Metadata_Object_Count).all(), 'Count disagreement')
            g = g[g.composto.isin(['DMSO', 'EOS100315'])].copy()
            require(len(g) == 29 and g.composto.eq('DMSO').sum() == 28, '28 DMSO + one Alda-1 required')
            g['site'] = site
            g = g.merge(key[['site', 'replica', 'poco', 'sample_id', 'papel_ficha']],
                on=['site', 'replica', 'poco'], how='left', validate='many_to_one')
            g['papel_ficha'] = g.papel_ficha.fillna('DMSO nao exibido')
            g['sample_id'] = g.sample_id.fillna('')
            dm = g.composto.eq('DMSO')
            pg = pars[pars.replica.eq(rep)].set_index('medida')
            require(len(pg) == len(FEATURES), 'Normalization parameter absent')
            for feat in FEATURES:
                p = pg.loc[feat]
                ref = g.loc[dm, feat].to_numpy(float)
                med = float(np.median(ref))
                mad = float(1.4826 * np.median(np.abs(ref - med)))
                require(np.isclose(med, p.mediana_DMSO, rtol=1e-08, atol=1e-12) and np.isclose(mad,
                    p.MAD_escalado, rtol=1e-08, atol=1e-12), 'Saved MAD parameters do not match raw')
                valid = bool(p.escala_valida) and np.isfinite(mad) and (mad > 1e-18)
                zn = (g[feat].to_numpy(float) - float(p.mediana_DMSO)) / (float(p.MAD_escalado) + 1e-18) if valid else np.full(len(g),
                    np.nan)
                ref_z = (ref - float(p.mediana_DMSO)) / (float(p.MAD_escalado) + 1e-18) if valid else np.full(28,
                    np.nan)
                obj, ch = (feat.split('_')[0], feat.rsplit('_', 1)[1])
                for pos, (_, r) in enumerate(g.iterrows()):
                    v = float(r[feat])
                    pct = 100 * (np.sum(ref < v) + 0.5 * np.sum(ref == v)) / len(ref)
                    rank = float(pd.Series(ref).rank(method='average').iloc[np.flatnonzero(ref == v)[0]]) if r.composto == 'DMSO' else np.nan
                    values.append({'site': site, 'replica': rep, 'placa': 'B1001', 'poco': r.poco,
                        'composto': r.composto, 'sample_id': r.sample_id, 'papel_ficha': r.papel_ficha,
                        'medida': feat, 'objeto': obj, 'canal': ch, 'valor_agregado_fonte': v,
                        'mediana_DMSO_congelada': float(p.mediana_DMSO), 'MAD_escalado_congelado': float(p.MAD_escalado),
                        'z_MAD': zn[pos], 'razao_mediana_DMSO': v / med if med != 0 else np.nan,
                        'percentil_empirico_DMSO': pct, 'posto_entre_28_DMSO': rank, 'DMSO_n': len(ref),
                        'DMSO_min': float(np.min(ref)), 'DMSO_Q1': float(np.quantile(ref, 0.25)),
                        'DMSO_Q3': float(np.quantile(ref, 0.75)), 'DMSO_max': float(np.max(ref)),
                        'escala_valida': valid, 'n_objetos': r.n_objetos, 'arquivo_fonte': raw_path.name,
                        'linha_csv': r.linha_csv})
            cols = ['site', 'placa', 'replica', 'poco', 'composto', 'dose', 'n_objetos',
                'objetos_DMSO_mediana', 'objetos_relativos_DMSO', 'sample_id', 'papel_ficha'] + SCORES
            wells.append(g[cols])
            sources.append({'site': site, 'replica': rep, 'arquivo': raw_path.name,
                'sha256': sha, 'n_perfis_arquivo': len(raw), 'n_DMSO': 28, 'n_Alda1': 1,
                'pack': pack.name})
            passed(f'{site}/{rep} IDs, row numbers, counts and 12 frozen intensity scales checked')


## 5. Produzir a tabela integrada

Os valores continuam separados por local e réplica. Uma razão de intensidade maior que 1 não é um novo hit.

In [ ]:
well = pd.concat(wells, ignore_index=True)
long = pd.DataFrame(values)
require(len(well) == 348 and len(long) == 4176, 'Unexpected output size')
sel = well[well.sample_id.ne('')].copy()
pp = plan.set_index('sample_id').sort_index()
ss = sel.set_index('sample_id').sort_index()
for c in SCORES + ['n_objetos']:
    require(np.allclose(pp[c].astype(float), ss[c].astype(float), rtol=1e-09,
        atol=1e-09, equal_nan=True), 'Plan numerical mismatch ' + c)
passed('All original selected scores unchanged and match current raw identities')
primary = long[long.medida.eq(PRIMARY)].copy()
alda = well[well.composto.eq('EOS100315')].merge(primary[primary.composto.eq('EOS100315')][['site',
    'replica', 'valor_agregado_fonte', 'z_MAD', 'razao_mediana_DMSO', 'percentil_empirico_DMSO']],
    on=['site', 'replica'], validate='one_to_one').merge(visual, on=['site',
    'replica'], validate='one_to_one')
for k in [1, 2]:
    ctrl = primary[primary.papel_ficha.eq(f'DMSO {k}')][['site', 'replica',
        'poco', 'sample_id', 'valor_agregado_fonte', 'z_MAD', 'posto_entre_28_DMSO']].copy()
    ctrl = ctrl.rename(columns={c: f'DMSO{k}_{c}' for c in ctrl if c not in ['site', 'replica']})
    alda = alda.merge(ctrl, on=['site', 'replica'], validate='one_to_one')
other = [c for c in SCORES if c != 'A_Mito']
alda['A_outros_max'] = alda[other].max(axis=1)
alda['bloco_outros_max'] = alda[other].idxmax(axis=1)
alda['site_order'] = alda.site.map({v: i for i, v in enumerate(SITES)})
alda = alda.sort_values(['site_order', 'replica']).drop(columns='site_order').reset_index(drop=True)
save('integracao_12_condicoes', alda)
save('intensidades_12_medidas_todos_DMSO', long)
save('escores_e_contagens_348_pocos', well)
save('intensidade_Mito_primaria_controles_selecionados', primary[primary.papel_ficha.str.startswith('DMSO ') & primary.sample_id.ne('')])
save('fontes_perfis_B1001_R1_R3', pd.DataFrame(sources))
save('conferencia_36_pocos', sel)


## 6. Conferir os diagnósticos dos painéis

Clipping de exibição pertence ao campo, enquanto o escore pertence ao perfil agregado do poço.

In [ ]:
diag = pd.read_csv(ROOT / 'diagnosticos_pixels.csv')
require(len(diag) == 432 and set(diag.replica) == set(REPS), 'Unexpected images or R4 present')
require(set(diag['campo']) == {1, 5, 9}, 'Field set changed')
require(diag.groupby(['sample_id', 'campo']).canal.nunique().eq(4).all(), 'Incomplete four-channel fields')
diag = diag.merge(key[['sample_id', 'composto', 'papel_ficha']], on='sample_id', validate='many_to_one')
save('diagnosticos_pixels_identificados', diag)
passed('432 image diagnostics link to key, R1-R3 fields 1,5,9, four channels')


## 7. Preservar as fontes e fechar

O resultado pode divergir da leitura humana; a divergência não autoriza substituir a resposta ou excluir um DMSO.

In [ ]:
require(all((digest(ROOT / name) == h for name, h in tracked.items())), 'Original inputs changed')
passed('All tracked input hashes unchanged after integration')
(OUT / 'auditoria' / 'manifesto.json').write_text(json.dumps({'scope': 'R1-R3; no R4 data accessed; no models trained',
    'run_sources': RUNS, 'features_defined_by_semantics_before_values': FEATURES,
    'primary_feature': PRIMARY, 'n_visual_rows': len(visual), 'n_control_wells': 336,
    'n_case_wells': 12, 'n_selected_control_wells': 24, 'source_hashes': tracked,
    'checks': tests}, ensure_ascii=False, indent=2), encoding='utf-8')
print('CHECKS', len(tests), 'PASSED')
print(alda[['site', 'replica', 'Mito_visual', 'A_Mito', 'A_outros_max',
    'bloco_outros_max', 'n_objetos', 'z_MAD', 'razao_mediana_DMSO', 'DMSO1_poco',
    'DMSO1_z_MAD', 'DMSO1_posto_entre_28_DMSO', 'DMSO2_poco', 'DMSO2_z_MAD',
    'DMSO2_posto_entre_28_DMSO']].to_string(index=False))


In [ ]:
display(alda[['site','replica','Mito_visual','A_Mito','razao_mediana_DMSO','n_objetos']])
print('Integração concluída:', OUT)
print('Nenhum modelo, critério, resposta humana ou dado de R4 foi alterado.')